In [8]:
import os
import re
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

AMP_FASTA = RAW_DIR / "amp.fasta"

NEGATIVE_FASTAS = [
    RAW_DIR / "namp_faba.fasta", 
    RAW_DIR / "namp_viri.fasta",
]

In [4]:
def parse_fasta(path):
    records = []
    header = None
    seq_chunks = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_chunks)))
                header = line[1:]
                seq_chunks = []
            else:
                seq_chunks.append(line)

        if header is not None:
            records.append((header, "".join(seq_chunks)))

    return records

records = parse_fasta(fasta_path)
len(records)

574627

In [5]:
df = pd.DataFrame(records, columns=["header", "sequence"])
df["length"] = df["sequence"].str.len()
df.head()

,header,sequence,length
0,sp|A0A009IHW8|ABTIR_ACIB9 2' cyclic ADP-D-ribo...,MSLEQKKGADIISKILQIQNSIGKTTSPSTLKTKLSEISRKEQENA...,269
1,"sp|A0A023I7E1|ENG1_RHIMI Glucan endo-1,3-beta-...",MRFQVIVAAATITMITSYIPGVASQSTSDGDDLFVPVSNFDPKSIF...,796
2,sp|A0A024B7W1|POLG_ZIKVF Genome polyprotein OS...,MKNPKKKSGGFRIVNMLKRGVARVSPFGGLKRLPAGLLLGHGPIRM...,3423
3,sp|A0A024RXP8|GUX1_HYPJR Exoglucanase 1 OS=Hyp...,MYRKLAVISAFLATARAQSACTLQSETHPPLTWQKCSSGGTCTQQT...,514
4,sp|A0A024SC78|CUTI1_HYPJR Cutinase OS=Hypocrea...,MRSLAILTTLLAGHAFAYPKPAPQSVNRRDWPSINEFLSELAKVMP...,248


In [6]:
STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def is_standard_protein(seq):
    return set(seq).issubset(STANDARD_AAS)

df["is_standard"] = df["sequence"].apply(is_standard_protein)
df["n_unique_chars"] = df["sequence"].apply(lambda s: len(set(s)))

print(df["is_standard"].value_counts())

is_standard
True     571959
False      2668
Name: count, dtype: int64


In [7]:
df = df[df["is_standard"]].copy()
df.reset_index(drop=True, inplace=True)
print(df.shape)

(571959, 5)
